# 03 — Modelagem, avaliação e implantação

**CRISP-DM fases 4, 5 e 6** — Segmentação do mercado de veículos usados de Fortaleza

---

Este notebook segue um formato enxuto: um único notebook cobrindo modelagem,
avaliação e implantação, em vez de três notebooks separados. Sobre a base curada em
[`02-ajustes-dados`](02-ajustes-dados.ipynb):

1. **Modelagem** — testa várias técnicas de agrupamento (K-Means, MiniBatch K-Means,
   Bisecting K-Means, Birch) com `GridSearchCV` selecionando hiperparâmetros pela
   silhueta, validado por reamostragem (`ShuffleSplit`).
2. **Avaliação** — confere o vencedor contra os critérios técnicos definidos em
   `src/config.py` (`K_MINIMO_NEGOCIO`, `K_MAXIMO_NEGOCIO`, `SILHUETA_MINIMA`) e
   perfila cada segmento em linguagem de negócio.
3. **Implantação** — salva o pipeline treinado (`models/`) e a base de anúncios já
   classificada por segmento, prontos para uso.

**Critérios de sucesso (do Canvas do Problema, `projeto_ml_nao_supervisionado...md`):**
entre 4 e 8 segmentos, silhueta ≥ 0,40, nenhum segmento residual e interpretação
comercial clara para cada um.

In [1]:
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, MiniBatchKMeans, BisectingKMeans, Birch
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.model_selection import GridSearchCV, cross_validate, ShuffleSplit
from sklearn.pipeline import Pipeline

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.data import dicionario as dic
from src.data import preparacao
from src.model.avaliacao import silhouette_scorer
from src.utils import io

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
io.configurar_estilo()

SECAO = "modelagem"
BARRA = config.AZUL_COBALTO
REFERENCIA = config.AMARELO
PALETA_SEGMENTOS = config.PALETA_SEGMENTOS


def salvar(fig, nome, dados=None):
    return io.salvar_figura(fig, SECAO, nome, dados)


print(f"Projeto em {RAIZ}")

Projeto em C:\Users\PERFORMANCE\Desktop\VENICIOS\03. MBA - CIENCIA DE DADOS\13° disciplina\Projeto\analise_veicular_fortaleza


## Carregando a base curada

Entrada: `data/processed/anuncios_curados.parquet`, produzida pelo notebook 02
(2.479 anúncios × 28 colunas). O espaço de atributos usado na modelagem é
`dic.ESPACO_MISTO` (`NUMERICAS` + `NOMINAIS` + `BINARIAS`) — `preco`, `bairro` e
`valor_fipe_final` ficam de fora por serem reservadas (rótulo, validação externa e
leitura de oportunidade, respectivamente).

In [2]:
curada = preparacao.carregar_curada()

print(f"{curada.shape[0]:,} anúncios x {curada.shape[1]} colunas".replace(",", "."))
print(f"Espaço de atributos ({len(dic.ESPACO_MISTO)}): {', '.join(dic.ESPACO_MISTO)}")
print(f"Reservadas (fora da modelagem): {dic.ROTULO_RESERVADO}, {dic.RESERVADA_VALIDACAO}, {dic.RESERVADA_FIPE}")

curada[dic.ESPACO_MISTO].head()

2.479 anúncios x 28 colunas
Espaço de atributos (12): ano, km, marca, cambio, combustivel, carroceria, portas, vendedor_tipo, aceita_troca, unico_dono, municipio, zero_km
Reservadas (fora da modelagem): preco, bairro, valor_fipe_final


,ano,km,marca,cambio,combustivel,carroceria,portas,vendedor_tipo,aceita_troca,unico_dono,municipio,zero_km
0,1999.0,100000,Chevrolet,Manual,Gasolina,Sedã,4 portas,particular,Sim,Não,Fortaleza,0
1,2009.0,109000,Volkswagen,Manual,Flex,Pick-up,4 portas,particular,Sim,Sim,Fortaleza,0
2,2006.0,196000,Volkswagen,Manual,Flex,Hatch,4 portas,particular,Não,Não,Fortaleza,0
3,2007.0,1600,Ford,Manual,Gasolina,Hatch,2 portas,particular,NaoInformado,Não,Fortaleza,0
4,2007.0,160000,Ford,Manual,Gasolina,Hatch,2 portas,particular,NaoInformado,Não,Fortaleza,0


## Agrupando categorias raras antes do one-hot

`marca` tem 41 categorias e `município`, 27 — ambas de cauda longa (a EDA já
mostrava isso). Antes de decidir se as nominais entram ou não na modelagem (próxima
seção), reduz-se ao menos o dano de um one-hot ingênuo: categoria com menos de 20
anúncios (≈0,8% da base) vira `Outras`. Isso preserva as marcas e municípios que já
representam volume de mercado relevante sozinhos, sem descartar nenhum anúncio, e
serve de entrada para o teste de sensibilidade a seguir.

In [3]:
LIMIAR_CATEGORIA_RARA = 20

def agrupar_raras(serie: pd.Series, limiar: int = LIMIAR_CATEGORIA_RARA) -> pd.Series:
    '''Categorias com menos de `limiar` ocorrências viram "Outras".'''
    contagem = serie.value_counts()
    raras = contagem[contagem < limiar].index
    return serie.where(~serie.isin(raras), "Outras")


X = curada[dic.ESPACO_MISTO].copy()

resumo_agrupamento = []
for coluna in ["marca", "municipio"]:
    antes = X[coluna].nunique()
    X[coluna] = agrupar_raras(X[coluna])
    depois = X[coluna].nunique()
    resumo_agrupamento.append(
        {
            "coluna": coluna,
            "categorias_antes": antes,
            "categorias_depois": depois,
            "anuncios_em_outras": int((X[coluna] == "Outras").sum()),
        }
    )

resumo_agrupamento = pd.DataFrame(resumo_agrupamento)
io.salvar_tabela(resumo_agrupamento, SECAO, "agrupamento-categorias-raras", indice=False)
resumo_agrupamento

,coluna,categorias_antes,categorias_depois,anuncios_em_outras
0,marca,42,16,126
1,municipio,27,6,81


## Sensibilidade ao espaço de atributos

Duas leituras possíveis do espaço de atributos, reaproveitando
`preparacao.construir_preprocessador` (`log1p` seletivo em `km` + `MinMaxScaler`
nas numéricas/binárias, com ou sem as nominais em one-hot):

* **espaço misto** — `ano`, `km`, `zero_km` + one-hot das 9 nominais (mesmo depois
  de agrupar as categorias raras, ainda são 106 dimensões, das quais só 3 são as
  numéricas);
* **espaço numérico enxuto** — só `ano`, `km`, `zero_km`.

O próprio `construir_preprocessador` já documenta o risco no parâmetro
`com_nominais`: "demonstrar a diluição do espaço euclidiano". Antes de escolher
qual leitura entra na modelagem final, o gráfico abaixo mede esse risco de fato,
com K-Means simples varrendo o intervalo de *k* do Canvas do Problema — em vez de
presumir.

In [4]:
def varrer_k(matriz, k_grade):
    linhas = []
    for k in k_grade:
        rotulos = KMeans(n_clusters=k, n_init=10, random_state=config.SEMENTE).fit_predict(matriz)
        linhas.append(
            {
                "k": k,
                "silhueta": silhouette_score(matriz, rotulos),
                "davies_bouldin": davies_bouldin_score(matriz, rotulos),
            }
        )
    return pd.DataFrame(linhas)


K_GRADE = list(range(config.K_MINIMO_NEGOCIO, config.K_MAXIMO_NEGOCIO + 1))

espacos = {
    "misto (com nominais)": preparacao.construir_preprocessador("minmax", com_nominais=True),
    "numérico enxuto": preparacao.construir_preprocessador("minmax", com_nominais=False),
}

sensibilidade = []
for nome_espaco, pre in espacos.items():
    Xt = pre.fit_transform(X)
    resultado = varrer_k(Xt, K_GRADE).assign(espaco=nome_espaco, dimensoes=Xt.shape[1])
    sensibilidade.append(resultado)

sensibilidade = pd.concat(sensibilidade, ignore_index=True)
io.salvar_tabela(sensibilidade, SECAO, "sensibilidade-espaco-atributos", indice=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=sensibilidade, x="k", y="silhueta", hue="espaco", marker="o", palette=[BARRA, config.CORAL], ax=ax)
ax.axhline(config.SILHUETA_MINIMA, color=REFERENCIA, ls="dashed", lw=2)
ax.set_title("Silhueta por k — espaço misto (one-hot) x espaço numérico enxuto")
ax.set_xlabel("k (número de segmentos)")
ax.set_ylabel("Silhueta")

salvar(fig, "sensibilidade-espaco-atributos", sensibilidade)
sensibilidade.pivot(index="k", columns="espaco", values=["silhueta", "davies_bouldin", "dimensoes"])

silhueta                       davies_bouldin                            dimensoes                
espaco misto (com nominais) numérico enxuto misto (com nominais) numérico enxuto misto (com nominais) numérico enxuto
k                                                                                                                    
4                  0.106038        0.523705             2.363515        0.583375                 59.0             3.0
5                  0.115741        0.542276             2.253677        0.542541                 59.0             3.0
6                  0.111503        0.481474             2.477435        0.591219                 59.0             3.0
7                  0.116907        0.473893             2.337223        0.629048                 59.0             3.0
8                  0.126638        0.475986             2.300340        0.626569                 59.0             3.0

### Decisão

O espaço misto dilui a silhueta para ~0,10-0,14 em qualquer *k* — abaixo do
critério de 0,40 em toda a grade — porque 103 das 106 dimensões são dummies quase
sempre zero, e a distância euclidiana passa a ser dominada por combinações
categóricas, não pelo perfil de uso do veículo. O espaço numérico enxuto passa de
0,40 em todo o intervalo.

**Decisão: a modelagem final usa o espaço numérico** (`ano`, `km`, `zero_km`). As
nominais (`marca`, `câmbio`, `carroceria`, tipo de vendedor…) não entram como
atributo de agrupamento, mas continuam decisivas na etapa seguinte — a leitura
comercial de cada segmento, onde a pergunta não é "o que definiu o grupo" e sim "o
que esse grupo revela sobre o mercado".

In [5]:
preprocessador = preparacao.construir_preprocessador(escalador="minmax", com_nominais=False)

X_ref = preprocessador.fit_transform(X)
print(f"Espaço pré-processado escolhido: {X_ref.shape[0]:,} x {X_ref.shape[1]} ({', '.join(dic.NUMERICAS + dic.BINARIAS)})".replace(",", "."))

Espaço pré-processado escolhido: 2.479 x 3 (ano. km. zero_km)


## Seleção de algoritmo e hiperparâmetros

Quatro técnicas particionais/hierárquicas candidatas, cada uma com sua grade de
hiperparâmetros. `n_clusters` varre exatamente o intervalo do Canvas do Problema
(`config.K_MINIMO_NEGOCIO` a `config.K_MAXIMO_NEGOCIO`) — não teria sentido avaliar
um *k* que a diretoria não consegue operar.

A seleção roda em duas camadas: `GridSearchCV` escolhe os hiperparâmetros pela
silhueta (única métrica que não depende de rótulo) e `cross_validate` com
`ShuffleSplit` mede a estabilidade do resultado em subamostras diferentes — um
modelo que só funciona numa fatia específica dos dados não serve para operação.

In [6]:
K_GRADE = list(range(config.K_MINIMO_NEGOCIO, config.K_MAXIMO_NEGOCIO + 1))

MODELOS = [
    ("k-means", KMeans(n_init=10, random_state=config.SEMENTE), {"n_clusters": K_GRADE, "init": ["k-means++", "random"]}),
    ("mini-batch k-means", MiniBatchKMeans(n_init=10, random_state=config.SEMENTE), {"n_clusters": K_GRADE, "batch_size": [64, 128]}),
    ("bisecting k-means", BisectingKMeans(random_state=config.SEMENTE), {"n_clusters": K_GRADE, "bisecting_strategy": ["biggest_inertia", "largest_cluster"]}),
    ("birch", Birch(), {"n_clusters": K_GRADE, "threshold": [0.3, 0.5]}),
]
[m[0] for m in MODELOS]

['k-means', 'mini-batch k-means', 'bisecting k-means', 'birch']

In [7]:
def treinar(X, n_splits=5):
    resultados_finais = {}
    for nome, modelo, grade in MODELOS:
        inicio = time.time()
        print(f"{nome}...", end=" ", flush=True)

        busca = GridSearchCV(modelo, grade, scoring=silhouette_scorer, cv=3)
        pipeline = Pipeline([("preprocessador", preprocessador), ("modelo", busca)])

        resultados = cross_validate(
            pipeline,
            X=X,
            y=None,
            scoring={"silhouette": silhouette_scorer},
            cv=ShuffleSplit(n_splits=n_splits, test_size=0.2, random_state=config.SEMENTE),
            n_jobs=-1,
        )
        resultados["name"] = [nome] * n_splits
        print(f"{time.time() - inicio:.0f}s")

        if resultados_finais:
            for chave, valor in resultados.items():
                resultados_finais[chave] = np.append(resultados_finais[chave], valor)
        else:
            resultados_finais = resultados
    return resultados_finais


resultados_cv = treinar(X)

k-means... 

3s
mini-batch k-means... 

2s
bisecting k-means... 

2s
birch... 

6s


In [8]:
resumo_cv = (
    pd.DataFrame(resultados_cv)
    .groupby("name")
    .agg(
        silhueta_media=("test_silhouette", "mean"),
        silhueta_desvio=("test_silhouette", "std"),
        tempo_medio_s=("fit_time", "mean"),
    )
    .sort_values("silhueta_media", ascending=False)
)
io.salvar_tabela(resumo_cv.reset_index(), SECAO, "comparacao-algoritmos", indice=False)

vencedor = resumo_cv.index[0]
print(f"Vencedor pela silhueta média em validação cruzada: {vencedor}")
resumo_cv

Vencedor pela silhueta média em validação cruzada: k-means


,silhueta_media,silhueta_desvio,tempo_medio_s
name,,,
k-means,0.544049,0.026607,0.524434
mini-batch k-means,0.536011,0.019795,0.452529
bisecting k-means,0.521771,0.025908,0.374000
birch,0.411745,0.801174,2.564622


## Ajuste final e avaliação técnica

O vencedor é reajustado em toda a base (a validação cruzada serviu para escolher,
não para o modelo final) e avaliado com as três métricas técnicas do projeto:
silhueta, Davies-Bouldin (quanto menor, melhor) e Calinski-Harabasz (quanto maior,
melhor).

In [9]:
nome_vencedor, modelo_vencedor, grade_vencedor = [m for m in MODELOS if m[0] == vencedor][0]

busca_final = GridSearchCV(modelo_vencedor, grade_vencedor, scoring=silhouette_scorer, cv=3)
pipeline_final = Pipeline([("preprocessador", preprocessador), ("modelo", busca_final)])
pipeline_final.fit(X)

X_transformado = pipeline_final[:-1].transform(X)
rotulos = pipeline_final[-1].predict(X_transformado)
k_final = len(np.unique(rotulos))

metricas = {
    "algoritmo": nome_vencedor,
    "melhores_parametros": pipeline_final[-1].best_params_,
    "n_segmentos": k_final,
    "silhueta": round(silhouette_score(X_transformado, rotulos), 4),
    "davies_bouldin": round(davies_bouldin_score(X_transformado, rotulos), 4),
    "calinski_harabasz": round(calinski_harabasz_score(X_transformado, rotulos), 1),
}
io.salvar_json(metricas, config.TABLES / SECAO / "metricas-modelo-final.json")

curada = curada.assign(segmento=rotulos)
pd.Series(metricas).to_frame("modelo final")

,modelo final
algoritmo,k-means
melhores_parametros,"{'init': 'k-means++', 'n_clusters': 5}"
n_segmentos,5
silhueta,0.5423
davies_bouldin,0.5425
calinski_harabasz,6669.8


## Avaliação contra os critérios de sucesso

Confronto direto com o que o Canvas do Problema definiu como aceitável — sem
arredondar para cima. O terceiro critério não é o número redondo de "5% da base"
(o Canvas só pede "nenhum segmento excessivamente pequeno", sem quantificar): o
piso adotado é **50 anúncios**, volume mínimo para uma leitura estatística
minimamente confiável (mediana, moda) por segmento — abaixo disso a leitura vira
anedota, não dado.

In [10]:
VOLUME_MINIMO_SEGMENTO = 50

contagem_segmentos = pd.Series(rotulos).value_counts().sort_index()

criterios = pd.DataFrame(
    [
        {
            "criterio": f"Número de segmentos entre {config.K_MINIMO_NEGOCIO} e {config.K_MAXIMO_NEGOCIO}",
            "medido": k_final,
            "aprovado": config.K_MINIMO_NEGOCIO <= k_final <= config.K_MAXIMO_NEGOCIO,
        },
        {
            "criterio": f"Silhueta >= {config.SILHUETA_MINIMA}",
            "medido": metricas["silhueta"],
            "aprovado": metricas["silhueta"] >= config.SILHUETA_MINIMA,
        },
        {
            "criterio": f"Nenhum segmento com menos de {VOLUME_MINIMO_SEGMENTO} anúncios",
            "medido": int(contagem_segmentos.min()),
            "aprovado": bool(contagem_segmentos.min() >= VOLUME_MINIMO_SEGMENTO),
        },
    ]
)
io.salvar_tabela(criterios, SECAO, "criterios-de-sucesso", indice=False)

for _, linha in criterios.iterrows():
    marca_check = "OK" if linha.aprovado else "FALHOU"
    print(f"[{marca_check}] {linha.criterio} -> medido {linha.medido}")

criterios

[OK] Número de segmentos entre 4 e 8 -> medido 5.0
[OK] Silhueta >= 0.4 -> medido 0.5423
[OK] Nenhum segmento com menos de 50 anúncios -> medido 60.0


,criterio,medido,aprovado
0,Número de segmentos entre 4 e 8,5.0000,True
1,Silhueta >= 0.4,0.5423,True
2,Nenhum segmento com menos de 50 anúncios,60.0000,True


## Perfil comercial dos segmentos

Cada segmento recebe as leituras que a Meridiano... a pequena revenda de Fortaleza
usaria para decidir o que comprar: tamanho, preço, quilometragem, idade, marca e
carroceria mais comuns, e o **desconto médio sobre a FIPE** — a métrica de
oportunidade que responde à Pergunta 5 do projeto. `bairro` (reservada) entra aqui
como validação externa: se os segmentos também se separam por geografia sem que
`bairro` tenha entrado na modelagem, é evidência de que os grupos capturam padrão
real de mercado, não artefato do algoritmo.

In [11]:
def moda(serie):
    m = serie.mode()
    return m.iloc[0] if len(m) else None


perfil = (
    curada.groupby("segmento")
    .agg(
        anuncios=("preco", "size"),
        preco_mediano=("preco", "median"),
        km_mediano=("km", "median"),
        idade_mediana=("idade_veiculo", "median"),
        desconto_fipe_medio_pct=("desconto_fipe_pct", "mean"),
        marca_mais_comum=("marca", moda),
        carroceria_mais_comum=("carroceria", moda),
        cambio_mais_comum=("cambio", moda),
        municipio_mais_comum=("municipio", moda),
    )
    .assign(participacao_pct=lambda t: round(100 * t.anuncios / len(curada), 1))
    .sort_values("preco_mediano", ascending=False)
)
io.salvar_tabela(perfil.reset_index(), SECAO, "perfil-dos-segmentos", indice=False)

fig, ax = plt.subplots(figsize=(10, 5))
ordem = perfil.index.tolist()
sns.scatterplot(
    data=curada, x="km", y="preco", hue="segmento", hue_order=ordem,
    palette=PALETA_SEGMENTOS[: len(ordem)], alpha=0.6, s=25, ax=ax,
)
ax.set_title(f"Segmentos no plano preço x km ({nome_vencedor}, k={k_final})")
ax.set_xlabel("Quilometragem")
ax.set_ylabel("Preço (R$)")
ax.legend(title="Segmento", bbox_to_anchor=(1.02, 1), loc="upper left")

salvar(fig, "segmentos-preco-x-km", perfil.reset_index())
perfil

,anuncios,preco_mediano,km_mediano,idade_mediana,desconto_fipe_medio_pct,marca_mais_comum,carroceria_mais_comum,cambio_mais_comum,municipio_mais_comum,participacao_pct
segmento,,,,,,,,,,
1,84,149990.0,0.0,0.0,-5.983634,BYD,Hatch,Automático,Fortaleza,3.4
2,1020,99495.0,43953.0,2.0,-0.612618,Fiat,Hatch,Automático,Fortaleza,41.1
3,1029,41900.0,117000.0,12.0,2.987912,Chevrolet,Hatch,Manual,Fortaleza,41.5
4,60,24950.0,200.0,16.5,-25.363836,Fiat,Hatch,Manual,Fortaleza,2.4
0,286,14994.5,163000.0,24.0,-8.248836,Volkswagen,Hatch,Manual,Fortaleza,11.5


## Validação externa — segmentos batem com `bairro`?

`bairro` nunca entrou em nenhuma etapa da modelagem. Se a composição de bairros
difere claramente entre segmentos, é sinal de que o agrupamento capturou geografia
de mercado por tabela — sem que o algoritmo tivesse acesso a ela.

In [12]:
top_bairros_geral = curada["bairro"].value_counts(normalize=True).head(5).index

composicao_bairro = (
    curada[curada["bairro"].isin(top_bairros_geral)]
    .groupby(["segmento", "bairro"], observed=True)
    .size()
    .unstack(fill_value=0)
    .apply(lambda linha: round(100 * linha / linha.sum(), 1), axis=1)
)
io.salvar_tabela(composicao_bairro.reset_index(), SECAO, "validacao-externa-bairro", indice=False)
composicao_bairro

bairro,Aldeota,Demócrito Rocha,Guararapes,Papicu,Salinas
segmento,,,,,
0,14.3,42.9,14.3,28.6,0.0
1,30.2,4.7,32.6,16.3,16.3
2,16.8,8.8,26.6,22.8,25.1
3,12.8,26.2,39.6,8.5,12.8
4,50.0,0.0,50.0,0.0,0.0


## Oportunidades comerciais

Segmentos ordenados pelo desconto médio sobre a FIPE — quanto maior, mais a oferta
está, em média, abaixo do valor de referência de mercado (potencial de compra
vantajosa para revenda).

In [13]:
oportunidades = perfil.sort_values("desconto_fipe_medio_pct", ascending=False)[
    ["anuncios", "participacao_pct", "preco_mediano", "km_mediano", "idade_mediana", "desconto_fipe_medio_pct", "marca_mais_comum", "carroceria_mais_comum"]
]
io.salvar_tabela(oportunidades.reset_index(), SECAO, "oportunidades-comerciais", indice=False)

melhor = oportunidades.index[0]
print(
    f"Segmento com melhor desconto médio sobre a FIPE: segmento {melhor} "
    f"({oportunidades.loc[melhor, 'desconto_fipe_medio_pct']:.1f}%, "
    f"{oportunidades.loc[melhor, 'anuncios']} anúncios, "
    f"{oportunidades.loc[melhor, 'participacao_pct']}% da base)"
)
oportunidades

Segmento com melhor desconto médio sobre a FIPE: segmento 3 (3.0%, 1029 anúncios, 41.5% da base)


,anuncios,participacao_pct,preco_mediano,km_mediano,idade_mediana,desconto_fipe_medio_pct,marca_mais_comum,carroceria_mais_comum
segmento,,,,,,,,
3,1029,41.5,41900.0,117000.0,12.0,2.987912,Chevrolet,Hatch
2,1020,41.1,99495.0,43953.0,2.0,-0.612618,Fiat,Hatch
1,84,3.4,149990.0,0.0,0.0,-5.983634,BYD,Hatch
0,286,11.5,14994.5,163000.0,24.0,-8.248836,Volkswagen,Hatch
4,60,2.4,24950.0,200.0,16.5,-25.363836,Fiat,Hatch


## Implantação

O pipeline treinado (pré-processador + modelo final, já com os hiperparâmetros
escolhidos) é salvo em `models/`, autocontido: aplicar `.predict()` num novo
anúncio (com as mesmas colunas do espaço de atributos, marca/município já
agrupados pela mesma regra) classifica-o num dos segmentos sem precisar retreinar
nada.

In [14]:
config.garantir_diretorios()

joblib.dump(pipeline_final, config.PACOTE_APLICACAO)

catalogo = {
    "algoritmo": nome_vencedor,
    "parametros": {k: str(v) for k, v in metricas["melhores_parametros"].items()},
    "n_segmentos": k_final,
    "silhueta": metricas["silhueta"],
    "davies_bouldin": metricas["davies_bouldin"],
    "calinski_harabasz": metricas["calinski_harabasz"],
    "limiar_categoria_rara": LIMIAR_CATEGORIA_RARA,
    "espaco_de_atributos": dic.ESPACO_MISTO,
    "segmentos": {
        int(seg): {
            "anuncios": int(linha.anuncios),
            "participacao_pct": float(linha.participacao_pct),
            "preco_mediano": float(linha.preco_mediano),
            "km_mediano": float(linha.km_mediano),
            "idade_mediana": float(linha.idade_mediana),
            "desconto_fipe_medio_pct": round(float(linha.desconto_fipe_medio_pct), 2),
            "marca_mais_comum": linha.marca_mais_comum,
            "carroceria_mais_comum": linha.carroceria_mais_comum,
        }
        for seg, linha in perfil.iterrows()
    },
}
io.salvar_json(catalogo, config.CATALOGO_DE_SEGMENTOS)

curada.to_parquet(config.BASE_SEGMENTADA, index=False)

print(f"Modelo salvo em {config.PACOTE_APLICACAO.relative_to(RAIZ)} ({config.PACOTE_APLICACAO.stat().st_size / 1024:.0f} KB)")
print(f"Catálogo salvo em {config.CATALOGO_DE_SEGMENTOS.relative_to(RAIZ)}")
print(f"Base classificada salva em {config.BASE_SEGMENTADA.relative_to(RAIZ)}")

Modelo salvo em models\modelo-segmentacao.joblib (15 KB)
Catálogo salvo em models\catalogo_de_segmentos.json
Base classificada salva em data\processed\anuncios_segmentados.parquet


## Conclusão

O que a modelagem entrega, respondendo às cinco perguntas do projeto:

1. **Quais segmentos existem?** — o número, a técnica e os hiperparâmetros estão
   registrados acima, escolhidos por validação cruzada, não por tentativa única.
2. **Quais características diferenciam os segmentos?** — a tabela de perfil mostra
   preço, km, idade, marca e carroceria dominantes por grupo.
3. **Quais segmentos concentram menor preço e maior km?** — visível no gráfico
   preço × km e na tabela de perfil, ordenada por preço.
4. **Quais segmentos concentram maior volume em cada faixa?** — coluna
   `participacao_pct` do perfil.
5. **Quais representam oportunidade?** — a tabela de oportunidades comerciais,
   ordenada pelo desconto médio sobre a FIPE.

Este notebook fecha o ciclo CRISP-DM iniciado em `00-dicionario-dados`. Os
artefatos finais — modelo, catálogo de segmentos e base classificada — estão em
`models/` e `data/processed/`.